<a href="https://colab.research.google.com/github/Aleeza-Nadeem/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aleeza-Nadeem/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This notebook audits my own Week-5 model the way the FlyRank paper was audited in the live session: where does the label come from, does the validation design carry the claim, and are my own claims worded safely. All numbers are observed / measured / directional, for decision support only.

## 1. Two paper findings + my methodology questions

**Paper:** FlyRank, *The State of AI-Driven SEO in Numbers* (March 2026): 341,701 content pieces across 57 brands. Its headline findings are direct aggregate comparisons; the machine-learning pages are an exploratory appendix, and the paper states that it is observational and reports no p-values or confidence intervals. I read it with that evidence standard in mind. The paper was written for a broad audience, so my goal is to practice rigor, not to grade it.

### Finding 1 — "The Freshness Multiplier" (Finding #4)
**What it reports:** pages older than 365 days that were refreshed within the last 30 days show a health score of 34.5 versus 10.7 for comparable stale pages, and average impressions of 4,039 versus 71 (3.2x and 57x). The paper calls refresh timing one of the strongest measured levers.

- **Where does the label come from?** The group is "refreshed within 30 days" versus not. I would like to understand how the comparison pages were chosen, because it would let me trust the 57x number more as an effect of refreshing and less as an effect of *which* pages get refreshed. If teams tend to refresh pages that are already gaining impressions, the outcome could be picking the group. Also, health score includes impressions (30 of 100 points), so the 3.2x and 57x are not two independent confirmations. The active-content subset requires impressions and sessions above zero, which can favour survivors (the paper itself flags this for one cell in Finding #8).
- **Does the validation design carry the claim?** The comparison is between two groups at one point in time: no before/after on the same pages, no held-out brands, no interval around the 57x. "Lever" is a causal word, and this design supports "associated with". I would also like to understand why the 365+ refreshed group shows health 34.5 in Finding #4 but 44.62 in the matching cell of Finding #8 (365+ days old, refreshed 0-30 days). If they use different subsets, that tells me how stable the number is.

### Finding 2 — "Click Capture by Position Tier" (Finding #3)
**What it reports:** weighted CTR falls from 0.423% for positions 1-3 to 0.339% (4-10), 0.325% (11-20), 0.163% (21-50) and 0.050% (50+), an 88% drop from the top tier to the deep tier. The paper says these are portfolio weights, not general Google CTR benchmarks.

- **Where does the label come from?** The tier is built from a page's average position over a 90-day window. A page averaging position 15 could have some queries at rank 3 and others at rank 40, so the tiers mix different rank experiences. Weighted CTR is total clicks over total impressions per tier, so a few very large pages can set a tier's value; the median page may look different. (My Week-4 baseline used the *median* CTR per position bucket, learned on training clients only, so the two choices answer different questions.) I would like to understand why page 1 (0.339%) and positions 11-20 (0.325%) are only about 4% apart, because it would let me trust the curve more as a rank effect and less as an averaging effect.
- **Does the validation design carry the claim?** The curve is a descriptive cross-section, which fits the paper's own caution. The playbook step that page-one refinement is a more reliable source of incremental clicks goes one step further: it needs a before/after on the revised pages against matched untouched pages. With 57 brands, I would also like to see whether the curve holds on brands left out of the estimate, which is the same question my grouped split asks of my own model.

*Respectful framing to keep:* "I would like to understand X better, because it would let me trust claim Y more."

**What I take into my own audit:** define the label from a later window than the features, hold out whole clients, compare against a simple baseline on the same split, and avoid causal verbs for observational numbers.


In [1]:
import pandas as pd, numpy as np, json, os
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

url = "https://huggingface.co/datasets/FlyRank/internship-starter/resolve/main/content_refresh_anonymized.csv"
df = pd.read_csv(url)
need = ["clicks_last_30d","impressions_prev_30d","clicks_prev_30d","sessions_prev_30d","avg_position","client_id"]
d = df.dropna(subset=need).copy()
d = d[d.impressions_prev_30d > 0].reset_index(drop=True)

num = ["impressions_prev_30d","clicks_prev_30d","sessions_prev_30d","search_volume","competition","cpc",
       "word_count","content_age_days","days_since_last_update","avg_position"]
cat = ["content_type","main_intent"]
# anonymize content_type labels so no brand/client-like names appear in outputs
ct = d["content_type"].fillna("unknown")
d["content_type"] = ct.map({v: f"type_{k+1}" for k, v in enumerate(ct.value_counts().index)})

X = d[num].copy()
for c in num: X[c] = X[c].fillna(X[c].median())
X = pd.concat([X, pd.get_dummies(d[cat].fillna("unknown"), dtype=float)], axis=1)

def p_at_k(y_true, score, k=100):
    idx = np.argsort(-np.asarray(score), kind="stable")[:k]
    return float(np.asarray(y_true)[idx].mean())
print("rows used:", len(d), "| clients:", d.client_id.nunique(), "| features:", X.shape[1])

rows used: 26612 | clients: 31 | features: 18


## 2. My model under an honest split (before / after)

**Before:** a random row split (pages from the same client can sit in both train and test).
**After:** a grouped split by `client_id` (whole clients held out). Same model, same features, same metric (Precision@100 and ROC AUC). If the honest number is lower, that is the useful finding.

In [2]:
def run(train_idx, test_idx, label):
    cut = d.loc[train_idx, "clicks_last_30d"].quantile(0.80)      # cut-off from TRAIN rows only
    y = (d.clicks_last_30d >= cut).astype(int)
    rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, n_jobs=-1,
                                random_state=42, class_weight="balanced")
    rf.fit(X.iloc[train_idx], y.iloc[train_idx])
    s = rf.predict_proba(X.iloc[test_idx])[:,1]
    yt = y.iloc[test_idx].values
    return {"split": label, "test_pages": len(test_idx),
            "precision_at_100": round(p_at_k(yt, s, 100),3), "roc_auc": round(roc_auc_score(yt, s),3)}, s, yt, rf

# BEFORE: random rows
idx = np.arange(len(d))
tr_r, te_r = train_test_split(idx, test_size=0.25, random_state=42)
r_before, *_ = run(tr_r, te_r, "random rows (before)")

# AFTER: grouped by client
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_g, te_g = next(gss.split(X, groups=d.client_id))
r_after, s_g, yt_g, rf_g = run(tr_g, te_g, "grouped by client (after)")

shared = set(d.client_id.iloc[tr_r]) & set(d.client_id.iloc[te_r])
print("clients shared between train and test in the random split:", len(shared))
print("clients shared in the grouped split:", len(set(d.client_id.iloc[tr_g]) & set(d.client_id.iloc[te_g])))
ba = pd.DataFrame([r_before, r_after])
ba

clients shared between train and test in the random split: 31
clients shared in the grouped split: 0


,split,test_pages,precision_at_100,roc_auc
0,random rows (before),6653,1.00,0.971
1,grouped by client (after),6026,0.96,0.975


**Before vs after:** Under a random row split, all 31 clients appeared in both train and test, and Precision@100 was 1.00 with ROC AUC 0.971. When whole clients were held out (0 shared), the numbers were 0.95 and 0.975. AUC barely moved, so the model does not seem to depend on client habits here. The small drop in Precision@100 is within what I would expect from noise, since the two test sets differ in size (6,653 vs 6,026 pages). These are measured, directional results.

In [3]:
# Persistence check on the same held-out clients and the same label: rank by last month's clicks
pers = d.clicks_prev_30d.iloc[te_g].values
print("Persistence (rank by clicks_prev_30d): precision_at_100 =", round(p_at_k(yt_g, pers, 100), 3),
      "| roc_auc =", round(roc_auc_score(yt_g, pers), 3))
print("Random Forest (grouped split):         precision_at_100 =", r_after["precision_at_100"],
      "| roc_auc =", r_after["roc_auc"])


Persistence (rank by clicks_prev_30d): precision_at_100 = 1.0 | roc_auc = 0.968
Random Forest (grouped split):         precision_at_100 = 0.96 | roc_auc = 0.975


**Persistence check:** on the same held-out clients, ranking pages by last month's clicks alone gives Precision@100 of 1.00 and ROC AUC of 0.968, about the same as the Random Forest. So the model's ranking skill is mostly persistence. This is the simple baseline I want any gain to be measured against, in the same way the paper's multipliers should be read against a plain comparison group.

## 3. Leakage audit

The same hunt as Week 3, on the final feature set: banned columns, suspiciously strong single features, a shuffled-label sanity check, and real failure examples.

In [4]:
# 1) banned columns: nothing from the label window, no rule-made flags, no health score
banned_words = ["last_30d","needs_","is_","ai_opportunity","health_score","trend","_tier"]
bad = [col for col in X.columns if any(w in col for w in banned_words)]
print("Banned-looking feature names in X:", bad if bad else "none")

# 2) any single feature that predicts the label too well on held-out clients?
cut = d.loc[tr_g, "clicks_last_30d"].quantile(0.80)
y_all = (d.clicks_last_30d >= cut).astype(int)
single = {}
for col in X.columns:
    v = X.iloc[te_g][col].values
    if np.nanstd(v) > 0:
        a = roc_auc_score(y_all.iloc[te_g], v)
        single[col] = max(a, 1-a)
single = pd.Series(single).sort_values(ascending=False)
print("\nStrongest single features (AUC on held-out clients; >0.95 would be suspicious):")
print(single.head(5).round(3))

# 3) shuffled-label check: should be near 0.5
rng = np.random.default_rng(0)
y_shuf = y_all.iloc[tr_g].sample(frac=1, random_state=0).values
rf_s = RandomForestClassifier(n_estimators=100, min_samples_leaf=20, n_jobs=-1, random_state=0).fit(X.iloc[tr_g], y_shuf)
print("\nShuffled-label AUC (expect about 0.5):", round(roc_auc_score(y_all.iloc[te_g], rf_s.predict_proba(X.iloc[te_g])[:,1]),3))

# 4) duplicated pages across train and test?
print("Duplicate content_id across train/test:", len(set(d.content_id.iloc[tr_g]) & set(d.content_id.iloc[te_g])))

Banned-looking feature names in X: none

Strongest single features (AUC on held-out clients; >0.95 would be suspicious):
clicks_prev_30d           0.968
impressions_prev_30d      0.961
sessions_prev_30d         0.930
content_age_days          0.676
days_since_last_update    0.641
dtype: float64

Shuffled-label AUC (expect about 0.5): 0.475
Duplicate content_id across train/test: 0


In [5]:
# Real failure examples (no ids): top-100 model picks that missed, and true top pages that were missed
t = d.iloc[te_g].copy(); t["score"] = s_g; t["hit"] = yt_g
top100 = t.sort_values("score", ascending=False).head(100)
fp = top100[top100.hit == 0]
cols = ["impressions_prev_30d","clicks_prev_30d","clicks_last_30d","avg_position"]
print("False positives in top 100:", len(fp), "of 100. Five examples:")
print(fp[cols].head(5).round(2))
miss = t[(t.hit == 1)].drop(top100.index, errors="ignore").sort_values("clicks_last_30d", ascending=False)
print("\nTrue top-20% pages the model ranked outside its top 100 (largest five):")
print(miss[cols].head(5).round(2))

False positives in top 100: 4 of 100. Five examples:
       impressions_prev_30d  clicks_prev_30d  clicks_last_30d  avg_position
2750                   5363               23                1           4.2
20150                  5328               26                4           4.1
15453                 36552               22                3           7.0
5226                   4495               19                1           4.3

True top-20% pages the model ranked outside its top 100 (largest five):
       impressions_prev_30d  clicks_prev_30d  clicks_last_30d  avg_position
17434                 52962              513              430           5.2
15198                 28026              298              232           3.4
5918                 218786              250              220           4.2
23549                 44828              172              193           4.9
20811                 84550              114              181           5.6


**Leakage audit result:** No banned columns were found in the features, no page appeared in both train and test, and the shuffled-label AUC was 0.501, close to chance, so the pipeline is not leaking through the split. The strongest single features, clicks_prev_30d (AUC 0.968) and impressions_prev_30d (0.961), went above the 0.95 warning mark. I read this as persistence (last month's clicks predict next month's clicks), not leakage, because both come from the earlier window. The honest weakness is that the task is mostly persistence. Known caveat: avg_position is a 90-day field that overlaps the last 30 days, so a mild window overlap remains. The failure examples (cell above) show the top picks that missed and the large pages that were under-ranked.

**Limits:** this is one grouped split with 8 held-out clients, so the numbers could move with a different split. Missing numeric values were filled with medians computed on all rows (a small, known shortcut; the Week-5 notebook fills from training rows only). All results are observed / measured / directional.


## 4. Claim rewrite

**My boldest sentence (before):** "The Random Forest finds the pages that will earn the most clicks, so refreshing them will raise traffic."

This goes further than the evidence in two ways: it implies the model predicts the future reliably, and it implies a refresh causes the gain. The numbers below are observed, not causal.

In [6]:
rf_row = r_after
safe_claim = (f"On held-out clients, the model's top 100 ranked pages included {rf_row['precision_at_100']:.0%} "
              f"that were in the top 20% for next-30-day clicks (measured on {rf_row['test_pages']:,} pages, "
              f"ROC AUC {rf_row['roc_auc']}). This is directional decision-support for choosing which pages to review "
              f"first. It does not show that refreshing a page causes more clicks.")
print(safe_claim)

On held-out clients, the model's top 100 ranked pages included 96% that were in the top 20% for next-30-day clicks (measured on 6,026 pages, ROC AUC 0.975). This is directional decision-support for choosing which pages to review first. It does not show that refreshing a page causes more clicks.


**My rewritten claim:** On held-out clients, the model's top 100 ranked pages included 95% that were in the top 20% for next-30-day clicks (measured on 6,026 pages, ROC AUC 0.975). This is directional decision-support for choosing which pages to review first. A simple rule that ranks by last month's clicks did about as well, so the result mostly reflects persistence. It does not show that refreshing a page causes more clicks.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.